In [ ]:
'''
Santa 2026 | Verified Frontier 127,985

Replay the pinned public portfolio and export the verified submission.
'''

In [ ]:
'''
Credits

The pinned programs come from Ertuğrul Özer's Multi-Engine Frontier release.
https://www.kaggle.com/code/ozertuu/santa-2026-multi-engine-frontier-lb-127-985

The calculator replay builds on Ertuğrul Özer's Starter Notebook.
https://www.kaggle.com/code/ozertuu/santa-2026-starter-notebook
'''

# Santa 2026 · Verified Frontier

**127,985 arithmetic instructions across 1,000 puzzles.** This notebook packages the public Multi-Engine Frontier portfolio and replays every program on the 48-register calculator. Compared with the previous 128,155 submission, it saves 170 instructions across 116 puzzles.

The portfolio is embedded as a checksum-verified snapshot. On Kaggle, attach the Santa 2026 competition data and run all cells to generate `submission.csv`.

## 1 · Inputs and configuration

In [ ]:
import csv
import hashlib
import os
import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from statistics import median
from tempfile import TemporaryDirectory
from time import perf_counter


COMPETITION = 'santa-2026'
PORTFOLIO_SHA256 = 'c731df4122d1a4e97aa66305a7903ecf33d32ae290b35e6bf011fffda6332e0e'
EXPECTED_SCORE = 127_985
SUBMISSION_COLUMNS = ['id', 'program']
MAX_INSTRUCTIONS = 4_096
MAX_PROGRAM_BYTES = 131_072

Instruction = tuple[str, int, int, int]
Program = tuple[Instruction, ...]


@dataclass(frozen=True)
class Config:
    puzzles_csv: Path
    output_csv: Path

    @classmethod
    def discover(cls) -> 'Config':
        kaggle = Path('/kaggle/input')
        if kaggle.is_dir():
            puzzles = cls.locate(kaggle, COMPETITION, 'puzzles.csv')
            return cls(puzzles, Path('/kaggle/working/submission.csv'))

        data = Path(os.environ.get('SANTA_DATA_DIR', 'data'))
        return cls(data / 'puzzles.csv',
                   Path(os.environ.get('SANTA_OUTPUT_CSV', 'submission.csv')))

    @staticmethod
    def locate(root: Path, slug: str, filename: str) -> Path:
        matches = [path for path in root.rglob(filename) if slug in path.parent.parts]
        if len(matches) != 1:
            raise FileNotFoundError(
                f'Expected one {slug}/{filename} input, found {len(matches)}'
            )
        return matches[0]

    def check_inputs(self) -> None:
        if not self.puzzles_csv.is_file():
            raise FileNotFoundError(
                f'Attach the {COMPETITION} competition data or copy puzzles.csv '
                f'to {self.puzzles_csv}'
            )

In [ ]:
'''
Embed the public portfolio so each run uses the same programs.
'''
import base64
import lzma


PUBLIC_PORTFOLIO_XZ = (
    '...'
    'WVo='
)

def pinned_portfolio() -> bytes:
    raw = lzma.decompress(base64.b64decode(PUBLIC_PORTFOLIO_XZ, validate=True))
    if hashlib.sha256(raw).hexdigest() != PORTFOLIO_SHA256:
        raise ValueError('The embedded public portfolio failed its checksum')
    return raw


## 2 · Exact calculator and program profile

In [ ]:
@dataclass(frozen=True)
class Puzzle:
    starts: tuple[int, ...]
    targets: frozenset[int]
    value_limit: int
    register_count: int

    @classmethod
    def from_row(cls, row: dict[str, str]) -> 'Puzzle':
        return cls(
            tuple(json.loads(row['starting_values'])),
            frozenset(json.loads(row['targets'])),
            int(row['value_limit']),
            int(row['registers']),
        )


@dataclass(frozen=True)
class ProgramProfile:
    length: int
    operations: Counter[str]
    peak_value: int
    populated_registers: int


class Calculator:
    '''Read both sources before writing and remember targets across overwrites.'''

    def replay(self, puzzle: Puzzle, program: Program) -> ProgramProfile:
        if len(program) > MAX_INSTRUCTIONS:
            raise ValueError('Program exceeds the instruction limit')

        registers = list(puzzle.starts) + [None] * (
            puzzle.register_count - len(puzzle.starts)
        )
        missing = set(puzzle.targets).difference(puzzle.starts)
        operations = Counter()
        peak_value = max(puzzle.starts, default=0)

        for number, instruction in enumerate(program, start=1):
            if len(instruction) != 4:
                raise ValueError(f'Step {number}: expected four instruction fields')
            op, left, right, dest = instruction
            if any(type(index) is not int or not 0 <= index < puzzle.register_count
                   for index in (left, right, dest)):
                raise ValueError(f'Step {number}: register index out of range')

            a, b = registers[left], registers[right]
            if a is None or b is None:
                raise ValueError(f'Step {number}: undefined source register')
            if op == '+':
                value = a + b
            elif op == '-':
                value = a - b
            elif op == '*':
                value = a * b
            elif op == '/' and a % b == 0:
                value = a // b
            else:
                raise ValueError(f'Step {number}: unsupported operation or inexact division')

            if not 1 <= value <= puzzle.value_limit:
                raise ValueError(f'Step {number}: result outside the value limit')
            registers[dest] = value
            missing.discard(value)
            operations[op] += 1
            peak_value = max(peak_value, value)

        if missing:
            raise ValueError(f'{len(missing)} missing targets')

        '''
        Defined registers stay populated after a write, so the final count
        is also the peak populated count for the program.
        '''
        return ProgramProfile(
            len(program), operations, peak_value,
            sum(value is not None for value in registers),
        )

## 3 · Validate the portfolio and export

In [ ]:
class PortfolioPipeline:
    def __init__(self, config: Config):
        self.config = config
        self.calculator = Calculator()

    def load_puzzles(self) -> dict[str, Puzzle]:
        with self.config.puzzles_csv.open(encoding='utf-8-sig', newline='') as file:
            rows = list(csv.DictReader(file))
        puzzles = {row['id']: Puzzle.from_row(row) for row in rows}
        expected = {f'scored_{index:04d}' for index in range(1, 1_001)}
        if len(rows) != len(expected) or puzzles.keys() != expected:
            raise ValueError('Expected the 1,000 scored puzzle IDs exactly once')
        return puzzles

    def verify(
        self, path: Path, puzzles: dict[str, Puzzle],
    ) -> tuple[dict[str, Program], dict[str, ProgramProfile]]:
        programs, profiles = {}, {}
        with path.open(encoding='utf-8', newline='') as file:
            reader = csv.reader(file)
            if next(reader, None) != SUBMISSION_COLUMNS:
                raise ValueError(f'{path}: expected the id,program header')
            for row in reader:
                if len(row) != 2:
                    raise ValueError(f'{path}: expected two fields at line {reader.line_num}')
                puzzle_id, cell = row
                if puzzle_id not in puzzles or puzzle_id in programs:
                    raise ValueError(f'{path}: unexpected or duplicate ID {puzzle_id}')
                if len(cell.encode('utf-8')) > MAX_PROGRAM_BYTES:
                    raise ValueError(f'{puzzle_id}: program cell exceeds the byte limit')
                try:
                    payload = json.loads(cell)
                    if type(payload) is not list or any(type(step) is not list for step in payload):
                        raise ValueError('Expected a JSON array of instruction arrays')
                    program = tuple(tuple(step) for step in payload)
                    profiles[puzzle_id] = self.calculator.replay(puzzles[puzzle_id], program)
                except ValueError as error:
                    raise ValueError(f'{puzzle_id}: {error}') from error
                programs[puzzle_id] = program
        if programs.keys() != puzzles.keys():
            raise ValueError(f'{path}: portfolio is missing scored puzzle IDs')
        return programs, profiles

    def export(self, programs: dict[str, Program], puzzles: dict[str, Puzzle]) -> None:
        output = self.config.output_csv
        output.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=output.parent) as directory:
            temporary = Path(directory) / output.name
            with temporary.open('w', encoding='utf-8', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(SUBMISSION_COLUMNS)
                for puzzle_id in sorted(programs):
                    cell = json.dumps(programs[puzzle_id], separators=(',', ':'))
                    writer.writerow((puzzle_id, cell))
            written, _ = self.verify(temporary, puzzles)
            if written != programs:
                raise RuntimeError('The exported programs differ from the verified input')
            temporary.replace(output)

    def run(self) -> dict[str, ProgramProfile]:
        started = perf_counter()
        self.config.check_inputs()
        puzzles = self.load_puzzles()
        with TemporaryDirectory() as directory:
            source = Path(directory) / 'our_programs.csv'
            source.write_bytes(pinned_portfolio())
            programs, profiles = self.verify(source, puzzles)
        lengths = [profile.length for profile in profiles.values()]
        score = sum(lengths)
        if score != EXPECTED_SCORE:
            raise ValueError(f'Unexpected portfolio score: {score:,}')
        self.export(programs, puzzles)

        operations = Counter()
        for profile in profiles.values():
            operations.update(profile.operations)
        print(f'Verified puzzles: {len(programs):,} | Local score: {score:,}')
        print(f'Program lengths: min {min(lengths)} | median {median(lengths):g} | max {max(lengths)}')
        print('Operation counts:', ', '.join(f'{op} {operations[op]:,}' for op in '+-*/'))
        print(f'CSV: {self.config.output_csv} | {perf_counter() - started:.2f}s')
        return profiles

## 4 · Self-check and run

In [ ]:
calculator = Calculator()
puzzle = Puzzle((3,), frozenset({1, 2, 5, 8}), 9, 48)
example = (('/', 0, 0, 1), ('-', 0, 1, 1), ('+', 0, 1, 2), ('+', 0, 2, 3))
profile = calculator.replay(puzzle, example)
assert profile == ProgramProfile(4, Counter({'+': 2, '-': 1, '/': 1}), 8, 4)

try:
    calculator.replay(puzzle, (('/', 0, 0, 1), ('/', 1, 0, 2)))
except ValueError as error:
    assert 'inexact division' in str(error)
else:
    raise AssertionError('Inexact division passed replay')

profiles = PortfolioPipeline(Config.discover()).run()

## 5 · Where the instructions go

The left panel groups the verified instruction cost by number of targets. The right panel shows counts of executed operations. Operation frequency describes the portfolio and is not a measure of how much an operation could save.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter


class PortfolioFigures:
    @staticmethod
    def draw(puzzles, profiles):
        orange, blue = '#d9480f', '#1c7ed6'
        with plt.rc_context({
            'figure.facecolor': 'white', 'axes.spines.top': False,
            'axes.spines.right': False, 'axes.edgecolor': '#dee2e6',
            'axes.titlelocation': 'left', 'axes.titleweight': 'bold',
            'text.color': '#212529', 'grid.color': '#e9ecef',
        }):
            fig, (cost, operations) = plt.subplots(1, 2, figsize=(10.5, 3.7), layout='constrained')
            bands = [(1, 9), (10, 19), (25, 29), (30, 32)]
            totals = [
                sum(profile.length for pid, profile in profiles.items()
                    if low <= len(puzzles[pid].targets) <= high)
                for low, high in bands
            ]
            assert sum(totals) == sum(profile.length for profile in profiles.values())
            counts = Counter()
            for profile in profiles.values():
                counts.update(profile.operations)

            def bars(ax, labels, values, colors, title):
                rectangles = ax.barh(labels, values, color=colors, height=.6)
                ax.bar_label(rectangles, labels=[f'{value:,}' for value in values], padding=5)
                ax.set_xlim(0, max(values)*1.24)
                ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
                ax.set_title(title, pad=12)
                ax.set_xlabel('Executed instructions')
                ax.grid(axis='x')
                ax.set_axisbelow(True)

            bars(cost, [f'{a}–{b} targets' for a, b in bands], totals,
                 [blue, blue, orange, orange], 'Cost by target count')
            bars(operations, ['Addition', 'Subtraction', 'Multiplication', 'Exact division'],
                 [counts[op] for op in '+-*/'], [blue, blue, blue, orange], 'Operation mix')
            plt.show()


PortfolioFigures.draw(PortfolioPipeline(Config.discover()).load_puzzles(), profiles)